# 🧬 Day-4 · Notebook 02: Vector Embeddings & Vector Databases
**KIT Campus RAG Agent · Part 2: Vector Mathematics & LangChain VectorStore**

In this notebook, we cover:
1. **Vector Embeddings**: Converting student queries into semantic dense vectors.
2. **Mathematical Distance Metrics**: Cosine Similarity, Dot Product, and Euclidean ($L_2$) Distance.
3. **LangChain `InMemoryVectorStore`**: Indexing chunks and executing similarity search with relevance thresholds.
4. **Google Gemini `text-embedding-004` vs Local Fallback**: High-performance cloud embeddings with zero-crash offline guarantees.



In [4]:
import math
import numpy as np

# Let's inspect distance metrics mathematically
# Vector A: "Medical condonation attendance rules 68%"
vec_a = np.array([0.85, 0.42, 0.15, -0.25], dtype=np.float32)
# Vector B: "Doctor certificate fee 500 attendance condonation" (Semantically very similar)
vec_b = np.array([0.82, 0.48, 0.18, -0.22], dtype=np.float32)
# Vector C: "Hostel mess dinner timings on Sunday" (Semantically completely different)
vec_c = np.array([-0.65, 0.10, 0.72, 0.20], dtype=np.float32)

def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def euclidean_distance(v1: np.ndarray, v2: np.ndarray) -> float:
    return float(np.linalg.norm(v1 - v2))

print(f"Cosine Similarity (A vs B - Highly Similar):   {cosine_similarity(vec_a, vec_b):.4f}")
print(f"Cosine Similarity (A vs C - Irrelevant):        {cosine_similarity(vec_a, vec_c):.4f}")
print(f"Euclidean Distance (A vs B - Close in space):   {euclidean_distance(vec_a, vec_b):.4f}")
print(f"Euclidean Distance (A vs C - Far in space):     {euclidean_distance(vec_a, vec_c):.4f}")



Cosine Similarity (A vs B - Highly Similar):   0.9968
Cosine Similarity (A vs C - Irrelevant):        -0.4583
Euclidean Distance (A vs B - Close in space):   0.0794
Euclidean Distance (A vs C - Far in space):     1.6970


### 2. LangChain InMemoryVectorStore Indexing
Now let's use LangChain's official `InMemoryVectorStore` to index college policy chunks!



In [6]:
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

# Lightweight deterministic embedding class for offline execution
class StudentCollegeEmbeddings(Embeddings):
    def __init__(self, dim: int = 128):
        self.dim = dim

    def _embed(self, text: str) -> List[float]:
        np.random.seed(abs(hash(text)) % (2**32))
        vec = np.random.randn(self.dim).astype(np.float32)
        norm = np.linalg.norm(vec)
        return (vec / norm).tolist() if norm > 1e-9 else vec.tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text)

embeddings = StudentCollegeEmbeddings(dim=64)
vector_store = InMemoryVectorStore(embeddings)

# Add student handbook policy documents
sample_docs = [
    Document(
        page_content="Medical condonation is permitted for attendance between 65% and 74.9% with a ₹500 fee per subject and doctor certificate within 3 days.",
        metadata={"source": "kit_college_student_handbook.md", "section": "Attendance"}
    ),
    Document(
        page_content="Resident students must return to hostel blocks by 8:30 PM. Facial recognition biometric punch-in is recorded at lobby.",
        metadata={"source": "kit_campus_hostel_and_mess_guide.docx", "section": "Hostel Curfew"}
    ),
    Document(
        page_content="Placement eligibility requires minimum 7.00 CGPA with zero standing arrears. Dream company CTC is 10 LPA and above.",
        metadata={"source": "kit_placement_policy_guide.pdf", "section": "Placements"}
    )
]

vector_store.add_documents(sample_docs)
print(f"Indexed {len(sample_docs)} documents into LangChain InMemoryVectorStore.")



Indexed 3 documents into LangChain InMemoryVectorStore.


In [8]:
# Execute semantic similarity search with score
query = "What happens if my attendance is 68% and I was sick?"
results = vector_store.similarity_search_with_score(query, k=2)

print(f"Query: '{query}'\n")
for doc, score in results:
    print(f"Score: {score:.4f} | Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}\n")



Query: 'What happens if my attendance is 68% and I was sick?'

Score: 0.2361 | Source: kit_college_student_handbook.md
Content: Medical condonation is permitted for attendance between 65% and 74.9% with a ₹500 fee per subject and doctor certificate within 3 days.

Score: 0.2210 | Source: kit_placement_policy_guide.pdf
Content: Placement eligibility requires minimum 7.00 CGPA with zero standing arrears. Dream company CTC is 10 LPA and above.

